© 2026 by Tamás Takács is licensed under CC BY-NC-SA 4.0. To view a copy of this license, visit https://creativecommons.org/licenses/by-nc-sa/4.0/

English translation managed by Tamás Takács. The translation was produced with AI assistance.

# Twitter Sentiment Analysis (30 points)


**Import the notebook into Colab and work there!**

**Contestant's name:**


John Intelligent registered on Twitter on a sudden impulse. He has not yet discovered emojis, so he can only infer from the text of the messages what sentiment a given tweet might express.
Help him determine the mood in which the messages were written!

---
# Preparations

**Guides to the tools needed to solve the task:**
1. [Pandas](https://pandas.pydata.org/docs/user_guide/10min.html)
2. [Pandas Dataframe](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
3. [Tokenization](https://medium.com/@utkarsh.kant/tokenization-a-complete-guide-3f2dd56c0682)
4. [Tokenization, Mapping and Padding](https://medium.com/@lokaregns/preparing-text-data-for-transformers-tokenization-mapping-and-padding-9fbfbce28028)
5. [PyTorch Training/Inference](https://pytorch.org/tutorials/beginner/introyt/trainingyt.html)
6. [PyTorch Datasets and DataLoaders](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html)
7. [Pytorch NN Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html)
8. [TorchText Tokenizer](https://pytorch.org/text/stable/data_utils.html)
9. [Stop Word](https://en.wikipedia.org/wiki/Stop_word)
10. [Logistic Regression](https://www.spiceworks.com/tech/artificial-intelligence/articles/what-is-logistic-regression/#:~:text=Logistic%20regression%20is%20a%20supervised%20machine%20learning%20algorithm%20that%20accomplishes,1%2C%20or%20true%2Ffalse.)
11. [Logistic Regression sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)

The **tweets** needed for the solution come from the [Sentiment140](https://www.kaggle.com/datasets/kazanova/sentiment140) dataset (1.6M tweets, Stanford).

Download: [Kaggle Sentiment140](https://www.kaggle.com/datasets/kazanova/sentiment140)

In [ ]:
# @title Installing Dependencies
!pip install pandas --quiet
!pip install torchtext --quiet

In [ ]:
# @title Downloading the Tweet Dataset (Sentiment140)
# The original Google Drive link is no longer available.
# Alternative: download it from Kaggle:
# https://www.kaggle.com/datasets/kazanova/sentiment140

# With the Kaggle CLI:
# !pip install -q kaggle
# !kaggle datasets download -d kazanova/sentiment140
# !unzip -qq sentiment140.zip

# Original (not available):
# !gdown -qq 1penca66caOrgagaQvrKCBDrCN_g1ZN35
# !unzip -qq twitter.zip -d .
# !rm -rf twitter.zip

## Required Libraries

We have imported a few libraries to get you started, but feel free to use any PyTorch-based tool if you need to. Please note that Keras and TensorFlow are **NOT ALLOWED** for solving this task!

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

import torchtext
from torchtext.data import get_tokenizer

from sklearn.utils import shuffle
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

## Illustrating the Twitter Dataset

The code below visualizes the size of the dataset and the first five tweets.

Each text also has a label indicating whether the sentiment (`target`) of the given tweet is **positive** (4) or **negative** (0).

[Pandas Dataframe](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)

In [ ]:
header_list = ["target", "id", "date", "query", "user", "text"]
df = pd.read_csv('training.1600000.processed.noemoticon.csv',
                 encoding = "ISO-8859-1", names=header_list)
print("Number of tweets:", len(df))
df.head()

# Task 1: Exploratory Data Analysis (14 Points)

In the first task we want to perform an initial analysis and data cleaning on a Twitter dataset in order to get to know the tweets better.

1. Transform the 'target' field so that instead of '0' and '4', the values '**0**' and '**1**' are used for the **negative** and **positive** sentiment tweets! (binary encoding) (1 point)

2. Split the dataset into **test** and **training** data! The split should be **20-80%**. The order of the data must not change during processing! (2 points)

3. Print out **5-5 pieces** of negative and positive sentiment-tweet pairs from the **training dataset**! (1 point)

4. Tokenize every tweet in **both the training and the test dataset**! Use torchtext's **get_tokenizer** function to tokenize the tweets! [TorchText Tokenizer](https://pytorch.org/text/stable/data_utils.html) (2 points)

5. Compute how many unique tokens the **training dataset** contains! (1 point)

6. Using a bar plot, illustrate the 100 most frequently used words and their number of occurrences in the **training dataset**! (2 points)

7. Illustrate the 100 least frequently used words and their number of occurrences in the **training dataset**! (2 points)

In [ ]:
def target_binary_encode():
    raise NotImplementedError("This function has not been implemented yet.")

def test_train_split():
    raise NotImplementedError("This function has not been implemented yet.")

def printing_five():
    raise NotImplementedError("This function has not been implemented yet.")

def tokenize_tweets():
    raise NotImplementedError("This function has not been implemented yet.")

def unique_token_count():
    raise NotImplementedError("This function has not been implemented yet.")

def top_contributing_words():
    raise NotImplementedError("This function has not been implemented yet.")

def top_hundred_words():
    raise NotImplementedError("This function has not been implemented yet.")

def bottom_hundred_words():
    raise NotImplementedError("This function has not been implemented yet.")

# Task 2: Data Cleaning (10 points)

As you could see from the results above, the number of unique words is very high. This is possible because the training tweets contain a lot of punctuation, stop words [Stop Word](https://en.wikipedia.org/wiki/Stop_word), various inflected forms of words and very rare words, and even the IDs of different users. (Stop words ('**stop words**') are the most frequently used words within a given language; in English, for example, these can be "a", "an", "and", "but", "with", and the various forms of the verb "to be" also belong here.)

Your tasks are the following:

1. Remove the **punctuation** from the training dataset! (1 point)

2. Remove the **stop words** from the training dataset! (2 points)

3. Remove from the training dataset every word that **occurs only once**! (2 points)

4. Remove all **user mentions** from the training dataset. (@USER_ID) (4 points)

5. Re-tokenize both the training and the test dataset. (1 point)

In [ ]:
def clean_dataset():
    raise NotImplementedError("This function has not been implemented yet.")

def retokenize_dataset():
    raise NotImplementedError("This function has not been implemented yet.")

# Task 3: Training a Classifier to Classify Tweets (6 points)

The last task is to carry out a logistic regression task. Logistic regression is a multivariate method that lets us categorize cases according to the categories of the dependent variable. [Logistic Regression](https://www.spiceworks.com/tech/artificial-intelligence/articles/what-is-logistic-regression/#:~:text=Logistic%20regression%20is%20a%20supervised%20machine%20learning%20algorithm%20that%20accomplishes,1%2C%20or%20true%2Ffalse.)


1. Using the sklearn package, use LogisticRegression with the ("saga") solver and apply it to the **training dataset**! [Logistic Regression sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) (2 points)

2. Using the predict function, test the model's performance on your **test set**! (Don't forget to **tokenize** it as well!) `LogisticRegression.predict()` (4 points)


Training the classifier above is **time-consuming** (a few minutes), so we recommend that you start working on the other tasks while the model is training.

In [ ]:
def train_classifier():
    raise NotImplementedError("This function has not been implemented yet.")


---
**You have reached the end of the task**. Download the finished notebook as follows:
```
File → Download → Download .ipynb
```
and then upload it, packed **(.zip)** **together with your other solutions**, to the CMS system.